# 02 · 自动求导（autograd）

这是 PyTorch 和 NumPy **最本质的区别**。

以前做数值计算，想求导数得自己推公式、自己写代码。PyTorch 不用——你只要把计算过程写出来，它会**自动帮你求导**。

深度学习的本质就是「不断求导 → 沿着梯度改参数」，所以这一节是理解后面一切的基础。

In [ ]:
import torch

# requires_grad=True 表示：请追踪这个变量，它需要求导
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2          # y = x²

y.backward()        # 反向传播，自动求导

# 数学上 dy/dx = 2x，当 x=3 时 = 6
print('x =', x.item())
print('y = x² =', y.item())
print('dy/dx =', x.grad, '  （2x = 2×3 = 6）')

## 1. 换个复杂点的函数试试

$$y = x^3 + 2x^2$$

手算的导数是 $\dfrac{dy}{dx} = 3x^2 + 4x$，当 $x=2$ 时应该是 $12 + 8 = 20$。

看 PyTorch 算得对不对。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 3 + 2 * x ** 2
y.backward()

print('y =', y.item())
print('dy/dx =', x.grad, '  （手算结果 20）')

In [ ]:
# 多个变量、更复杂的组合也一样自动搞定
a = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(2.5, requires_grad=True)

z = a * b + torch.sin(a)
z.backward()

print('z = a*b + sin(a) =', round(z.item(), 4))
print('∂z/∂a = b + cos(a) =', round(a.grad.item(), 4), '  手算:', round(2.5 + torch.cos(torch.tensor(1.5)).item(), 4))
print('∂z/∂b = a =', round(b.grad.item(), 4))

## 2. ⚠️ 梯度会累积（新手必踩的坑）

PyTorch 的 `.grad` 是**累加**的，不是覆盖。看下面这个例子。

In [ ]:
x = torch.tensor(1.0, requires_grad=True)

for i in range(3):
    y = x ** 2          # 每次 dy/dx 都是 2x = 2
    y.backward()
    print(f'第 {i+1} 次 backward 后，x.grad = {x.grad.item()}')

In [ ]:
# 正确做法：每轮手动清零
x = torch.tensor(1.0, requires_grad=True)

for i in range(3):
    if x.grad is not None:
        x.grad.zero_()      # ← 清零
    y = x ** 2
    y.backward()
    print(f'第 {i+1} 次 backward 后，x.grad = {x.grad.item()}')

print()
print('这就对了。记住：训练循环里必须清梯度。')

> 在后面训练神经网络时，你会看到 `optimizer.zero_grad()` —— 它干的就是这件事。
> **忘了清零，模型就学不对**，而且不报错，很难查。

## 3. 什么时候不需要求导：`with torch.no_grad()`

做**预测/推理**时我们不需要导数，关掉它可以省内存、跑得更快。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x * 3

print('y =', y.item())
print('y 需要梯度吗:', y.requires_grad, '  （False，说明追踪被关掉了）')

## 4. 梯度到底有什么用？—— 亲手做一次梯度下降

任务：找 $y = (x - 5)^2$ 的最小值。

显然答案是 $x = 5$。但我们**假装不知道**，用梯度下降一步步逼近它：

1. 算当前点的梯度（它指向「上升最快的方向」）
2. 往**反方向**走一小步（所以叫「下降」）
3. 重复

这就是所有神经网络的训练原理，只是把 $x$ 换成了几百万个参数。

In [ ]:
x = torch.tensor(0.0, requires_grad=True)   # 从 0 开始瞎猜
lr = 0.2                                     # 学习率：每步走多大

print('step      x        loss')
print('-' * 28)

for step in range(21):
    loss = (x - 5) ** 2
    loss.backward()
    
    with torch.no_grad():
        x -= lr * x.grad        # 往梯度的反方向走一步
        x.grad.zero_()          # 清梯度
    
    if step % 4 == 0 or step == 20:
        print(f'{step:4d}   {x.item():7.4f}   {loss.item():8.4f}')

print('-' * 28)
print(f'最终 x = {x.item():.4f}   （真值 5.0）')

看到没——从头到尾我们**没有推导过任何导数公式**，PyTorch 全自动算好了。

如果把 `lr` 改成 `2.0`（步子迈太大）再跑一次，你会看到它直接飞出去不收敛。**这就是学习率调不对的后果**，以后你一定会遇到。

In [ ]:
# 试试把学习率改大，看会发生什么
x = torch.tensor(0.0, requires_grad=True)
lr = 2.0        # ← 改成 2.0 试试，对比上面 0.2 的结果

for step in range(10):
    loss = (x - 5) ** 2
    loss.backward()
    with torch.no_grad():
        x -= lr * x.grad
        x.grad.zero_()
    print(f'step {step:2d}   x = {x.item():12.2f}')

print('\n看到了吗？学习率太大就会这样来回震荡甚至发散。')

## 动手练习

1. 求 $y = \sin(x) \cdot x$ 在 $x = 1.0$ 处的导数（先自己用求导法则算一遍，再验证）。
   提示：$(uv)' = u'v + uv'$，答案是 $\cos(1) + \sin(1) \approx 1.3818$。
2. 用梯度下降找 $y = (x - 3)^2 + (x + 1)^2$ 的最小值（先展开手算出理论答案，再用代码逼近）。
3. 把上面学习率那段改成 `lr = 0.01`，看看要多少步才能接近 5.0。

In [ ]:
# 在这里写你的答案


---

## 这一节要记住的

- `requires_grad=True` → 追踪这个变量；`.backward()` → 自动求导；`.grad` → 取导数
- **梯度是累加的**，每轮必须 `zero_()` 或 `optimizer.zero_grad()`
- 推理/预测时用 `with torch.no_grad():` 省资源
- 梯度下降 = 反复「算梯度 → 反方向走一步」，学习率决定步子大小

下一节 → **03 · 第一个神经网络**，把这两节的东西串起来。